# Prerequisites: Data Preparation & Baseline Evaluation (~15 min on 2x H200)

This notebook has two goals:
1. **Prepare the distillation dataset** — download WikiText-103 and tokenize it into the binary format expected by Megatron-Bridge. This dataset is used during the distillation step (after pruning) in all scenario notebooks.
2. **Establish the teacher baseline** — evaluate the original Qwen3-8B on MMLU before any compression.

> **Why prepare the dataset before compression?** The distillation step (which comes *after* pruning) requires a pre-tokenized dataset in Megatron binary format. Preparing it upfront avoids interrupting the compression pipeline and ensures a consistent dataset across all scenarios.

> **Note on calibration datasets:** Pruning also requires calibration data to score the importance of each component — the model runs forward passes on a small dataset to measure how much each neuron, head, or layer contributes to the output. This calibration data (we use `nvidia/Nemotron-Post-Training-Dataset-v2`) is handled separately in each scenario notebook. Minitron downloads it automatically under the hood, while Puzzletron requires an explicit preparation step. See the respective notebooks (`scenario1_minitron.ipynb`, `scenario2_puzzletron.ipynb`, etc.) for details.

**Prerequisites:** Run this notebook from inside the NeMo container with the base model already downloaded at `/workspace/models/Qwen3-8B` (see the guide's Prerequisites section).

## 0. Install ModelOpt from the cloned repository

Replace the container's pre-installed ModelOpt with the version from the cloned repository (which includes the Puzzletron and Minitron features used in this guide).

In [ ]:
!rm -rf /opt/venv/lib/python3.12/site-packages/modelopt
!cp -r /workspace/Model-Optimizer/modelopt /opt/venv/lib/python3.12/site-packages/modelopt
!mkdir -p /workspace/datasets

## 1. Download distillation dataset

For distillation, we use WikiText-103 — a small, generic language modeling dataset. We download it and save it as JSONL, which is the input format expected by the Megatron tokenizer.

In [ ]:
import json
import os
from datasets import load_dataset

DATA_PATH = '/workspace/datasets/wikitext-103-v1'
os.makedirs(DATA_PATH, exist_ok=True)

dataset = load_dataset('wikitext', 'wikitext-103-v1', split='train')

with open(f'{DATA_PATH}/wikitext-train.jsonl', 'w') as file:
    file.writelines(json.dumps(item) + '\n' for item in dataset)

print(f'Saved {len(dataset)} samples to {DATA_PATH}/wikitext-train.jsonl')

## 2. Tokenize for Megatron

Megatron-Bridge requires data in a pre-tokenized binary format (`.bin` + `.idx` index file) rather than raw text. This format allows fast random access during training without re-tokenizing on the fly.

This step runs the Qwen3-8B tokenizer over the WikiText-103 JSONL and writes the output in the Megatron binary format expected by the distillation scripts.

In [ ]:
from modelopt.torch.utils.plugins import megatron_preprocess_data

megatron_preprocess_data(
    jsonl_paths='/workspace/datasets/wikitext-103-v1/wikitext-train.jsonl',
    output_dir='/workspace/datasets/wikitext-103-v1',
    tokenizer_name_or_path='/workspace/models/Qwen3-8B',
    json_keys=['text'],
    workers=32,
    log_interval=100000,
)

## 3. Verify

Check that all expected files are in place.

In [ ]:
import os

distill_path = '/workspace/datasets/wikitext-103-v1'
expected_files = ['wikitext-train.jsonl', 'wikitext-train_text.bin', 'wikitext-train_text.idx']

print(f'Distillation dataset: {distill_path}')
for f in expected_files:
    full = os.path.join(distill_path, f)
    exists = os.path.isfile(full)
    size = os.path.getsize(full) / (1024**2) if exists else 0
    print(f'  {f}: {"OK" if exists else "MISSING"} ({size:.1f} MB)')

print()
print('Data preparation complete. You can now proceed to the scenario notebooks.')

## 4. Evaluate teacher model (baseline)

Before compressing, we establish the baseline MMLU score for the original Qwen3-8B. Results in the guide are expressed as a percentage of this number.

We use [`lm-eval`](https://github.com/EleutherAI/lm-evaluation-harness) — a standard open-source evaluation harness — to run the MMLU benchmark. MMLU (Massive Multitask Language Understanding) covers 57 subjects across STEM, humanities, and social sciences, using 4-choice questions. The 5-shot setting provides 5 in-context examples per question, which is the standard configuration for comparing LLMs on this benchmark.

In [ ]:
!lm_eval --model hf \
    --model_args pretrained=/workspace/models/Qwen3-8B,dtype=bfloat16 \
    --tasks mmlu \
    --num_fewshot 5 \
    --batch_size 4

**Expected result:** MMLU (5-shot) = **0.7493**. This is the teacher baseline used throughout the guide.